# Billings Preprocessing

In [1]:
import pandas as pd
import numpy as np

In [4]:
# =============================================================================
# BILLINGS TABLE - FINAL PREPROCESSING
# =============================================================================
# Unit of analysis : one row = one renewal cycle = one training sample
# Unique key       : Co_Ref + Prospect_Renewal_Date  →  cycle_key
# Target variable  : churn_label (0 = Won, 1 = Churned)
# =============================================================================

df = pd.read_csv("../../dataset/raw/billings.csv") 
print(f"Original shape: {df.shape}")

C:\Users\pragi\AppData\Local\Temp\ipykernel_14236\2944698860.py:9: DtypeWarning: Columns (15,16,19,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../dataset/raw/billings.csv")


Original shape: (122082, 59)


In [5]:
# =============================================================================
# STEP 1: DROP USELESS COLUMNS
# =============================================================================
cols_to_drop = [
    'Last_Years_Date_Paid',   # 100% null
    'Last_Payment_Score',     # always 0, zero variance
    'Last_Discount_Score',    # always 0, zero variance
    'Billings_Contractor',    # always 1, constant flag
    'DateTime_Out',           # exact duplicate of Renewal_Month
    'Current_Anchor_List',    # 100% null
]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f"After dropping useless columns: {df.shape}")

After dropping useless columns: (122082, 56)


In [6]:
# =============================================================================
# STEP 2: FIX DATE COLUMNS  (stored as DD-MM-YYYY strings)
# =============================================================================
date_cols = [
    'Renewal_Month',
    'Proforma_Date',
    'Registration_Date',
    'Prospect_Renewal_Date',
    'Closed_Date',
    'Last_Renewal',
]
date_cols = [c for c in date_cols if c in df.columns]
 
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%d-%m-%Y', errors='coerce')
    n_null = df[col].isna().sum()
    if n_null > 0:
        print(f"  {col}: {n_null} nulls after date parsing")

  Proforma_Date: 304 nulls after date parsing
  Registration_Date: 1018 nulls after date parsing
  Closed_Date: 8188 nulls after date parsing
  Last_Renewal: 48291 nulls after date parsing


In [7]:
# =============================================================================
# STEP 3: FIX DATA TYPES
# =============================================================================
 
# --- TRUE/FALSE booleans -> 1/0 integer ---
for col in ['Proforma_Auto_Renewal', 'Proforma_World_Pay_Token']:
    if col in df.columns:
        df[col] = df[col].map(
            {True: 1, False: 0, 'TRUE': 1, 'FALSE': 0,
             'true': 1, 'false': 0, 1: 1, 0: 0}
        ).astype('Int64')
 
# --- y/n string flags -> 1/0 integer ---
for col in ['Current_Auto_Renewal_Flag', 'Current_World_Pay_Token']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower().map({'y': 1, 'n': 0})
        df[col] = df[col].astype('Int64')
 
# --- Fix casing inconsistency in Proforma_Account_Stage ---
# 'vetting' and 'Vetting' are the same value
if 'Proforma_Account_Stage' in df.columns:
    df['Proforma_Account_Stage'] = (
        df['Proforma_Account_Stage'].str.strip().str.title()
    )
 
# --- Strip whitespace from band and group columns ---
for col in ['Band', 'Last_Band', 'Tenure_Group', 'Connection_Group', 'Anchor_Group']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
 
print("Dtypes fixed.")

Dtypes fixed.


In [8]:
# =============================================================================
# STEP 4: FIX DATA ERRORS
# =============================================================================
 
# --- Renewal_Year = 2050: data entry errors ---
bad_year = df['Renewal_Year'] == 2050
if bad_year.sum() > 0:
    print(f"\nFixing {bad_year.sum()} rows with Renewal_Year = 2050")
    df.loc[bad_year, 'Renewal_Year'] = (
        df.loc[bad_year, 'Prospect_Renewal_Date'].dt.year
    )
 
# --- Recompute Tenure_Years where missing but Registration_Date exists ---
mask = df['Tenure_Years'].isna() & df['Registration_Date'].notna()
if mask.sum() > 0:
    df.loc[mask, 'Tenure_Years'] = (
        (df.loc[mask, 'Prospect_Renewal_Date'] -
         df.loc[mask, 'Registration_Date']).dt.days / 365.25
    ).round(1)
    print(f"Recomputed Tenure_Years for {mask.sum()} rows")
 
print("Data errors fixed.")


Fixing 3 rows with Renewal_Year = 2050
Data errors fixed.


In [9]:
# =============================================================================
# STEP 5: CHURN LABEL  (from Prospect_Outcome)
# =============================================================================
# Won     -> 0 (retained)
# Churned -> 1 (churned)
# Open    -> exclude (outcome unknown, not used in training)
#
# Prospect_Status contradictions (e.g. "No Longer Trading" + Won) are NOT
# manually fixed here. They are kept as-is and handled naturally during
# feature encoding (target encoding will down-weight anomalies automatically)
 
print(f"\nProspect_Outcome:\n{df['Prospect_Outcome'].value_counts(dropna=False)}")
print(f"\nProspect_Status (top 20):\n{df['Prospect_Status'].value_counts().head(20)}")
 
# --- Drop pure noise rows from Prospect_Status ---
noise_statuses = [
    'Duplicate Entry',
    'Do Not Recognise Client',
    'Existing Safecontractor Member',
    'Existing Safecontractor Member Already Anchored',
    'Existing Safemark Member',
    'Campaign on Hold',
]
before = len(df)
df = df[~df['Prospect_Status'].isin(noise_statuses)].copy()
print(f"\nDropped {before - len(df)} noise rows from Prospect_Status")
 
# --- Keep only labelled rows ---
df = df[df['Prospect_Outcome'].isin(['Won', 'Churned'])].copy()
df['churn_label'] = (df['Prospect_Outcome'] == 'Churned').astype(int)
 
print(f"\nAfter removing Open/noise rows: {df.shape}")
print(f"Churn rate        : {df['churn_label'].mean():.2%}")
print(f"Churn distribution:\n{df['churn_label'].value_counts()}")


Prospect_Outcome:
Prospect_Outcome
Won        101226
Churned     12668
Open         8188
Name: count, dtype: int64

Prospect_Status (top 20):
Prospect_Status
Renewed                                94826
Application and Money In                5897
Renewal Proforma Sent                   5219
No Longer Trading                       2387
Do Not Work for Client                  2377
Not Value for Money                     2196
Refused to Discuss                      1180
Competitor Accreditation                 984
Attempted Contact                        925
Intention to Proceed                     722
Non Responsive                           698
Not Affordable                           612
Contact Made                             576
Existing Safecontractor Member           436
Cannot Pass Audit                        413
Not to be contacted                      409
Contact Made Deciding                    375
Potential Lost Account                   266
Promised to Pay - Fees not Rece

In [10]:
# =============================================================================
# STEP 6: PROSPECT_STATUS GROUPING  (feature, not label)
# =============================================================================
# Encodes WHERE the customer was in the renewal journey at scoring time.
# Encoding strategy (target / ordinal / one-hot) decided during feature eng.
# Contradictions left as-is — model handles them naturally.
 
status_group_map = {
    # Retained
    'Renewed':                                          'Retained',
    'Application and Money In':                         'Retained',
    'Paid':                                             'Retained',
    'Payment En Route':                                 'Retained_Pending',
    'Intention to Proceed':                             'Retained_Pending',
    'Intention to Proceed (including auto email)':      'Retained_Pending',
 
    # At risk — still engaging
    'Contact Made Deciding':                            'At_Risk_Engaging',
    'Need Time To Consider < 30 days':                  'At_Risk_Engaging',
    'Need time to consider':                            'At_Risk_Engaging',
    'Promised to Pay - Fees not Received':              'At_Risk_Engaging',
    'Invoiced':                                         'At_Risk_Engaging',
    'Part Payment':                                     'At_Risk_Engaging',
    'Potential Lost Account':                           'At_Risk_Engaging',
 
    # At risk — disengaging
    'Refused to Discuss':                               'At_Risk_Disengaging',
    'Non Responsive':                                   'At_Risk_Disengaging',
    'Attempted Contact':                                'At_Risk_Disengaging',
    'Contact Made':                                     'At_Risk_Disengaging',
    'Need Time To Consider > 30 days':                  'At_Risk_Disengaging',
    'Need time to consider - 60 days':                  'At_Risk_Disengaging',
    'Not to be contacted':                              'At_Risk_Disengaging',
    'Renewal Proforma Sent':                            'At_Risk_Disengaging',
 
    # Churned — price/value driven
    'Not Value for Money':                              'Churned_Price',
    'Not Affordable':                                   'Churned_Price',
    'Price Increase':                                   'Churned_Price',
    'Insufficient Contract Value':                      'Churned_Price',
 
    # Churned — business/external reasons
    'No Longer Trading':                                'Churned_Business',
    'Anchor client close down':                         'Churned_Business',
    'Outside UK':                                       'Churned_Business',
    'Do Not Work for Client':                           'Churned_Business',
    'Client Exemption':                                 'Churned_Business',
    'Lost Anchor Client':                               'Churned_Business',
    'Authority no longer a client':                     'Churned_Business',
    'Supply Only':                                      'Churned_Business',
    'Insufficient Work from Authorities':               'Churned_Business',
 
    # Churned — competitor
    'Competitor Accreditation':                         'Churned_Competitor',
 
    # Churned — service
    'Poor Customer Service':                            'Churned_Service',
 
    # Churned — accreditation
    'Cannot Pass Audit':                                'Churned_Accreditation',
    'Decline 3rd party accreditation requests':         'Churned_Accreditation',
 
    # Churned — disengaged
    'Not Interested':                                   'Churned_Disengaged',
}
 
df['Prospect_Status_Group'] = (
    df['Prospect_Status']
    .map(status_group_map)
    .fillna('Unknown')
)
 
print(f"\nProspect_Status_Group distribution:")
print(df['Prospect_Status_Group'].value_counts())


Prospect_Status_Group distribution:
Prospect_Status_Group
Retained                 100718
Churned_Business           5069
Churned_Price              3011
At_Risk_Disengaging        2509
Churned_Competitor          984
Churned_Accreditation       509
At_Risk_Engaging            305
Churned_Service             154
Retained_Pending             17
Unknown                      13
Churned_Disengaged            2
Name: count, dtype: int64


In [11]:
# =============================================================================
# STEP 7: HANDLE MISSING VALUES
# =============================================================================
 
# --- Connection columns: 94% missing = no connections ---
for col in ['Connection_Net', 'Connection_Qty',
            'Starting_Connection_Net', 'Starting_Connection_Qty']:
    if col in df.columns:
        df[col] = df[col].fillna(0)
 
# --- Discount_Amount: 90% missing = no discount applied ---
if 'Discount_Amount' in df.columns:
    df['Discount_Amount'] = df['Discount_Amount'].fillna(0)
 
# --- Last_Years_Price: missing = new customer (first renewal) ---
# Flag it first, then fill with current Amount as neutral baseline
df['is_new_customer'] = df['Last_Years_Price'].isna().astype(int)
df['Last_Years_Price'] = df['Last_Years_Price'].fillna(df['Amount'])
 
# --- Payment_Timeframe: missing = unknown timing ---
df['payment_timeframe_missing'] = df['Payment_Timeframe'].isna().astype(int)
df['Payment_Timeframe'] = df['Payment_Timeframe'].fillna(0)
 
# --- Last renewal history: missing for new customers ---
df['Last_Band']           = df['Last_Band'].fillna('New_Customer')
df['Last_Connections']    = df['Last_Connections'].fillna(0)
df['Last_Total_Net_Paid'] = df['Last_Total_Net_Paid'].fillna(0)
 
# --- Proforma flags: ~15% missing ---
# Impute from current equivalents where missing
for proforma_col, current_col in [
    ('Proforma_Auto_Renewal',    'Current_Auto_Renewal_Flag'),
    ('Proforma_World_Pay_Token', 'Current_World_Pay_Token'),
]:
    if proforma_col in df.columns and current_col in df.columns:
        mask = df[proforma_col].isna()
        df.loc[mask, proforma_col] = df.loc[mask, current_col]
 
print("Missing values handled.")

Missing values handled.


In [12]:
# =============================================================================
# STEP 8: AUDIT STATUS GROUPING
# =============================================================================
# 31 granular values -> 7 meaningful buckets
 
audit_status_map = {
    'Accredited':                                             'Accredited',
    'Accredited - Ssip Due To Expire':                        'Accredited',
    'Accred Due To Expire':                                   'Accredited',
    'Renewal Questionnaire Sent':                             'Renewal_In_Progress',
    'Renewal Questionnaire Received':                         'Renewal_In_Progress',
    '1St Renewal Questionnaire Reminder':                     'Renewal_In_Progress',
    'Final Renewal Questionnaire Reminder':                   'Renewal_In_Progress',
    'Renewal Report Sent - Awaiting Information':             'Renewal_In_Progress',
    'Renewal Additional Information Received':                'Renewal_In_Progress',
    '1St Reminder - Renewal Additional Info':                 'Renewal_In_Progress',
    'Final Reminder - Renewal Additional Info':               'Renewal_In_Progress',
    'Failed- Renewal Questionnaire Not Received':             'Renewal_Failed',
    'Failed- Renewal Additional Information Not Received':    'Renewal_Failed',
    'Initial Questionnaire Sent':                             'Initial_In_Progress',
    'Initial Questionnaire Received':                         'Initial_In_Progress',
    'Initial - 1St Questionnaire Reminder Sent':              'Initial_In_Progress',
    '1St Reminder Initial Add Info':                          'Initial_In_Progress',
    'Final Reminder (Initial) Additional Information Not Sent':'Initial_In_Progress',
    'Report Sent - Awaiting Information':                     'Initial_In_Progress',
    'Additional Information Received':                        'Initial_In_Progress',
    'Failed- Initial Questionnaire Not Received':             'Initial_Failed',
    'Failed- Initial Additional Info Not Received':           'Initial_Failed',
    'Failed- Accreditation Removed Ssip Exp':                 'Suspended_Failed',
    'Suspended - Ssip Expired':                               'Suspended_Failed',
    'Suspended - 1St Reminder Ssip Expired':                  'Suspended_Failed',
    'Suspended - 2Nd Reminder Ssip Expired':                  'Suspended_Failed',
    'Ssip Expired - 1St Reminder':                            'Suspended_Failed',
    'Suspended - Insurance Out-Of-Date':                      'Suspended_Failed',
    'Withdrawn From Scheme':                                  'Withdrawn',
    'Unknown':                                                'Unknown',
}
 
if 'Proforma_Audit_Status' in df.columns:
    df['Audit_Status_Group'] = (
        df['Proforma_Audit_Status']
        .str.strip().str.title()
        .map(audit_status_map)
        .fillna('Unknown')
    )
 
print(f"\nAudit_Status_Group distribution:")
print(df['Audit_Status_Group'].value_counts())


Audit_Status_Group distribution:
Audit_Status_Group
Accredited             72134
Renewal_In_Progress    12801
Renewal_Failed          9542
Unknown                 9131
Initial_Failed          6427
Initial_In_Progress     2369
Suspended_Failed         885
Withdrawn                  2
Name: count, dtype: int64


In [13]:
# =============================================================================
# STEP 9: FINAL CHECKS
# =============================================================================
print("\n" + "="*60)
print("BILLINGS PREPROCESSING COMPLETE")
print("="*60)
print(f"Final shape       : {df.shape}")
print(f"Churn rate        : {df['churn_label'].mean():.2%}")
print(f"\nChurn distribution:\n{df['churn_label'].value_counts()}")
 
print(f"\nRemaining missing values (top 20):")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(20))
 
new_cols = [
    'cycle_key', 'churn_label', 'Prospect_Status_Group',
    'price_delta', 'price_delta_pct',
    'discount_applied', 'discount_value',
    'paid_late', 'paid_early', 'fully_sticky',
    'has_connections', 'connection_revenue_share',
    'days_proforma_to_renewal', 'band_changed',
    'is_new_customer', 'payment_timeframe_missing',
    'Audit_Status_Group',
]
print(f"\nNew columns added:")
for col in new_cols:
    if col in df.columns:
        print(f"  + {col}")


BILLINGS PREPROCESSING COMPLETE
Final shape       : (113291, 61)
Churn rate        : 10.71%

Churn distribution:
churn_label
0    101163
1     12128
Name: count, dtype: int64

Remaining missing values (top 20):
Last_Renewal                  46398
Total_Net_Paid                12128
Proforma_Account_Stage         9129
Proforma_Audit_Status          9129
Tenure_Years                    967
Registration_Date               967
Proforma_Date                   240
Proforma_Approved_Lists          61
Renewal_Score_At_Release         61
Proforma_Membership_Status       61
#_of_Connection                  61
dtype: int64

New columns added:
  + churn_label
  + Prospect_Status_Group
  + is_new_customer
  + payment_timeframe_missing
  + Audit_Status_Group


In [16]:
# =============================================================================
# STEP 10: SAVE
# =============================================================================
df.to_csv("../../dataset/preprocessed/billings_preprocessed.csv", index=False)
print(f"\nSaved : billings_preprocessed.csv")
print(f"\nJoin columns exposed for interaction tables:")
print(f"  Filter   : call_date < Prospect_Renewal_Date (pre-renewal window)")


Saved : billings_preprocessed.csv

Join columns exposed for interaction tables:
  Filter   : call_date < Prospect_Renewal_Date (pre-renewal window)
